# NB3 — Feature Engineering

Computes 8 rolling statistics per sensor over a 60-second backward-looking window. Key features: `nan_count` (computed BEFORE NaN fill — captures intermittent_dropout), `bias_dev` (normalized deviation from training baseline — captures bias and drift). Fits StandardScaler on train split only. Saves `data/features.parquet`, `data/scaler.joblib`, `data/feature_cols.json`.

In [1]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 80)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

WINDOW_SIZE  = 60   # seconds
MIN_PERIODS  = 30   # minimum rows in window before computing stats

df = pd.read_parquet(DATA_DIR / "wadi_faulted.parquet")
sensor_ref  = json.loads((DATA_DIR / "sensor_cols.json").read_text())
SENSOR_COLS = sensor_ref["sensor_cols"]

print(f"Loaded: {df.shape}")
print(f"Sensor columns: {len(SENSOR_COLS)}")
print(f"\nLabel counts: { {int(k):int(v) for k,v in df['label'].value_counts().sort_index().items()} }")


Loaded: (806852, 106)
Sensor columns: 98

Label counts: {0: 786402, 1: 9977, 2: 10473}


## 1. Training Statistics for bias_dev (fit on train split only)

In [2]:
# Assign block IDs to the FULL dataset first (train + test combined).
# This is critical: if we compute block IDs per-split, isolated 30-row test
# windows appear as separate blocks and rolling windows never accumulate
# enough rows (min_periods=30), making nearly all test rolling features NaN.
# With full-dataset block IDs, all rows within a recording day share a block,
# so test windows within the same day form one long sequence for rolling.
timestamps_sorted = df.sort_values("timestamp")["timestamp"]
time_diffs = timestamps_sorted.diff().dt.total_seconds().fillna(0)
is_new_block = time_diffs > WINDOW_SIZE  # gap > 60s → new block
df["block_id"] = is_new_block.cumsum().astype(int).reindex(df.index)

n_blocks = df["block_id"].nunique()
print(f"Total blocks (gaps > {WINDOW_SIZE}s): {n_blocks}")
print(f"Block row counts:")
for bid, grp in df.groupby("block_id"):
    print(f"  Block {bid:>2}: {len(grp):>7,} rows  "
          f"{grp['timestamp'].min().date()} → {grp['timestamp'].max().date()}")

# Per-sensor mean and std on ALL training rows (train split, all labels).
# Matches the old working pipeline — using all training rows (not just normal)
# for a stable baseline that accounts for the mix of labels in training data.
train_mask = df["split"] == "train"
train_sensor_means = df.loc[train_mask, SENSOR_COLS].mean().to_dict()
train_sensor_stds  = df.loc[train_mask, SENSOR_COLS].std().to_dict()
n_valid_bias = sum(1 for s in train_sensor_stds.values() if s > 1e-6)
print(f"\nTraining rows for baseline stats: {train_mask.sum():,}")
print(f"Sensors eligible for bias_dev (train std > 1e-6): {n_valid_bias}")


Total blocks (gaps > 60s): 14
Block row counts:
  Block  0:  16,595 rows  2017-09-25 → 2017-09-25
  Block  1:  73,501 rows  2017-09-26 → 2017-09-26
  Block  2:  69,606 rows  2017-09-27 → 2017-09-27
  Block  3:  73,950 rows  2017-09-28 → 2017-09-28
  Block  4:  47,175 rows  2017-09-29 → 2017-09-29
  Block  5:  22,235 rows  2017-10-02 → 2017-10-02
  Block  6:  71,818 rows  2017-10-03 → 2017-10-03
  Block  7:  73,548 rows  2017-10-04 → 2017-10-04
  Block  8:  75,376 rows  2017-10-05 → 2017-10-05
  Block  9:  72,721 rows  2017-10-06 → 2017-10-06
  Block 10:  62,873 rows  2017-10-07 → 2017-10-07
  Block 11:  16,914 rows  2017-10-09 → 2017-10-09
  Block 12:  74,998 rows  2017-10-10 → 2017-10-10
  Block 13:  55,542 rows  2017-10-11 → 2017-10-11

Training rows for baseline stats: 636,979
Sensors eligible for bias_dev (train std > 1e-6): 98


## 2. Rolling Feature Computation

Stats per sensor: mean, std, min, max, rate_of_change, OLS_slope, nan_count, bias_dev.

**Important:** `nan_count` is computed from the raw data BEFORE any NaN fill. This is the key signal for detecting intermittent_dropout faults.

In [3]:
def compute_rolling_features(
    df_split: pd.DataFrame,
    sensor_cols: list[str],
    window: int,
    min_periods: int,
    train_means: dict,
    train_stds: dict,
) -> pd.DataFrame:
    """
    Compute 8 rolling statistics per sensor using full-dataset block IDs.
    Groups by block_id so test rows within the same recording day accumulate
    a proper rolling window rather than being isolated per-split blocks.
    nan_count is computed on the raw (un-filled) series to capture dropouts.
    """
    from numpy.lib.stride_tricks import sliding_window_view

    _x = np.arange(window, dtype=np.float64) - (window - 1) / 2.0
    _Sxx = float((_x ** 2).sum())
    _sw = (_x / _Sxx).astype(np.float64)

    def _slope_kernel(s: np.ndarray) -> float:
        m = len(s)
        if m < 2:
            return np.nan
        if m == window:
            return float(np.dot(_sw, s))
        x = np.arange(m, dtype=np.float64) - (m - 1) / 2.0
        Sxx = float((x ** 2).sum())
        return 0.0 if Sxx == 0 else float(np.dot(x / Sxx, s))

    feature_frames = []

    for col in sensor_cols:
        series = df_split[col].astype("float64")

        mean = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).mean()
        ).rename(f"{col}__mean_{window}s")
        std = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).std()
        ).rename(f"{col}__std_{window}s")
        mn = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).min()
        ).rename(f"{col}__min_{window}s")
        mx = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).max()
        ).rename(f"{col}__max_{window}s")
        roc = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).apply(
                lambda w: (w[-1] - w[0]) / window, raw=True
            )
        ).rename(f"{col}__roc_{window}s")
        slope = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).apply(
                _slope_kernel, raw=True
            )
        ).rename(f"{col}__slope_{window}s")

        # nan_count on raw (un-filled) series — captures intermittent_dropout
        nan_indicator = series.isna().astype("float64")
        nan_count = nan_indicator.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=1).sum()
        ).rename(f"{col}__nan_count_{window}s")

        feature_frames.extend([mean, std, mn, mx, roc, slope, nan_count])

        # bias_dev: (rolling_mean − train_mean) / train_std
        t_std = train_stds.get(col, 0.0)
        if t_std is not None and t_std > 1e-6:
            t_mean = train_means.get(col, 0.0)
            bias_dev = pd.Series(
                (mean.values - t_mean) / t_std,
                index=mean.index,
                name=f"{col}__bias_dev_{window}s",
                dtype="float32",
            )
            feature_frames.append(bias_dev)

    return pd.concat(feature_frames, axis=1).astype("float32")

print("Rolling feature function defined.")


Rolling feature function defined.


This can take ~ 10 min


In [4]:
import gc

feature_splits = []

for split in ["train", "test"]:
    print(f"\nProcessing {split} split...")
    df_split = df[df["split"] == split].sort_values("timestamp").reset_index(drop=True)
    print(f"  Rows: {len(df_split):,}")

    feat_df = compute_rolling_features(
        df_split, SENSOR_COLS, WINDOW_SIZE, MIN_PERIODS,
        train_sensor_means, train_sensor_stds
    )

    fault_cols = [c for c in df_split.columns if c.startswith("fault_")]
    meta_cols  = ["timestamp", "split", "label", "block_id"] + fault_cols

    out_df = pd.concat([
        df_split[meta_cols].reset_index(drop=True),
        pd.DataFrame(df_split[SENSOR_COLS].values, columns=SENSOR_COLS),
        feat_df.reset_index(drop=True),
    ], axis=1)

    feature_splits.append(out_df)
    print(f"  Feature columns: {len(feat_df.columns)}")
    gc.collect()

df_features = pd.concat(feature_splits, ignore_index=True)
print(f"\nFull feature frame: {df_features.shape}")



Processing train split...
  Rows: 636,979
  Feature columns: 784

Processing test split...
  Rows: 169,873
  Feature columns: 784

Full feature frame: (806852, 891)


## 3. NaN Fill & Feature Column Assembly

In [5]:
ROLLING_COLS = [c for c in df_features.columns
                if c.endswith(("_60s",)) and c not in SENSOR_COLS]

print(f"Rolling feature columns: {len(ROLLING_COLS)}")
print(f"NaN before fill: {df_features[ROLLING_COLS].isna().sum().sum():,}")

# Primary fill: use train-split column means (no leakage from test)
train_fill_means = df_features[df_features["split"] == "train"][ROLLING_COLS].mean()
df_features[ROLLING_COLS] = df_features[ROLLING_COLS].fillna(train_fill_means)

# Secondary fill with 0 for any column whose entire training split was NaN
# (e.g. bias_dev for a sensor with zero variance in normal training rows —
# std=0 makes bias_dev undefined; 0 = "no deviation" is the correct imputation)
remaining = df_features[ROLLING_COLS].isna().sum().sum()
if remaining > 0:
    cols_still_nan = [c for c in ROLLING_COLS if df_features[c].isna().any()]
    print(f"  {remaining:,} NaN remain in {len(cols_still_nan)} columns after primary fill "
          f"— filling with 0")
    df_features[ROLLING_COLS] = df_features[ROLLING_COLS].fillna(0)

# Raw sensor NaN fill
raw_nan = df_features[SENSOR_COLS].isna().sum().sum()
if raw_nan > 0:
    raw_fill_means = df_features[df_features["split"] == "train"][SENSOR_COLS].mean()
    df_features[SENSOR_COLS] = df_features[SENSOR_COLS].fillna(raw_fill_means)
    df_features[SENSOR_COLS] = df_features[SENSOR_COLS].fillna(0)  # fallback
    print(f"Filled {raw_nan:,} raw sensor NaNs")

FEATURE_COLS = SENSOR_COLS + ROLLING_COLS
print(f"\nTotal feature columns: {len(FEATURE_COLS)}")
print(f"  Raw:     {len(SENSOR_COLS)}")
print(f"  Rolling: {len(ROLLING_COLS)}")

# Verify no NaN/inf remain
assert df_features[FEATURE_COLS].isna().sum().sum() == 0, "NaN values remain"
assert not np.isinf(df_features[FEATURE_COLS].values).any(), "Inf values remain"
print("Feature matrix clean (no NaN, no inf)")


Rolling feature columns: 784
NaN before fill: 569,916
Filled 1,296 raw sensor NaNs

Total feature columns: 882
  Raw:     98
  Rolling: 784
Feature matrix clean (no NaN, no inf)


## 4. StandardScaler — Fit on Train, Apply to Both

Can take ~ 5 min

In [6]:
# Detect and drop zero-variance sensors before scaling.
# Fit a temporary scaler on a 100k training sample (cheap), then identify any
# sensor whose train-split scale_ < 1e-6 (constant/near-constant in training).
# Drop that sensor AND all its rolling variants — constant features add noise
# splits in the RF without contributing signal.  Matches old NB5 Cell 35.
train_mask = df_features["split"] == "train"
sample_idx = df_features.index[train_mask][:100_000]
X_sample = df_features.loc[sample_idx, FEATURE_COLS].values

scaler_detect = StandardScaler()
scaler_detect.fit(X_sample)
del X_sample

zero_std_base_sensors = set()
for i, std in enumerate(scaler_detect.scale_):
    if std < 1e-6:
        base = FEATURE_COLS[i].split("__")[0]
        zero_std_base_sensors.add(base)

zero_std_cols = [c for c in FEATURE_COLS if c.split("__")[0] in zero_std_base_sensors]
FEATURE_COLS  = [c for c in FEATURE_COLS if c not in zero_std_cols]
SENSOR_COLS   = [c for c in SENSOR_COLS  if c not in zero_std_base_sensors]
ROLLING_COLS  = [c for c in ROLLING_COLS if c not in zero_std_cols]

print(f"Zero-variance sensors dropped ({len(zero_std_base_sensors)}): {sorted(zero_std_base_sensors)}")
print(f"Features dropped: {len(zero_std_cols)}")
print(f"Remaining features: {len(FEATURE_COLS)}  (raw={len(SENSOR_COLS)}, rolling={len(ROLLING_COLS)})")

# Fit the real scaler on all training rows using the filtered feature set
X_train = df_features.loc[train_mask, FEATURE_COLS].values
X_all   = df_features[FEATURE_COLS].values

scaler = StandardScaler()
scaler.fit(X_train)

X_scaled = scaler.transform(X_all)
df_features[FEATURE_COLS] = X_scaled

print(f"\nScaler fit on {train_mask.sum():,} training rows")
print(f"Applied to {len(df_features):,} total rows")
print(f"Train feature mean (should be ~0): {X_scaled[train_mask].mean():.4f}")
print(f"Train feature std  (should be ~1): {X_scaled[train_mask].std():.4f}")


Zero-variance sensors dropped (8): ['1_MV_002_STATUS', '1_MV_003_STATUS', '1_P_006_STATUS', '2A_AIT_002_PV', '2_FIC_301_SP', '2_MCV_007_CO', '2_PIC_003_SP', '3_AIT_001_PV']
Features dropped: 72
Remaining features: 810  (raw=90, rolling=720)

Scaler fit on 636,979 training rows
Applied to 806,852 total rows
Train feature mean (should be ~0): 0.0000
Train feature std  (should be ~1): 0.9857


## 5. Save Artifacts

In [7]:
# Save feature parquet
feat_cols_keep = ["timestamp","split","label"] + FEATURE_COLS
# Also keep fault_type for per-fault evaluation in NB4
if "fault_type" in df_features.columns:
    feat_cols_keep = ["timestamp","split","label","fault_type"] + FEATURE_COLS

out_path = DATA_DIR / "features.parquet"
df_features[feat_cols_keep].to_parquet(out_path, index=False)
print(f"Saved: {out_path}  shape={df_features[feat_cols_keep].shape}")

# Save scaler
joblib.dump(scaler, DATA_DIR / "scaler.joblib")
print(f"Saved: {DATA_DIR / 'scaler.joblib'}")

# Save feature col reference
(DATA_DIR / "feature_cols.json").write_text(json.dumps({
    "sensor_cols":   SENSOR_COLS,
    "rolling_cols":  ROLLING_COLS,
    "feature_cols":  FEATURE_COLS,
    "window_size":   WINDOW_SIZE,
    "min_periods":   MIN_PERIODS,
    "n_features":    len(FEATURE_COLS),
}, indent=2))
print(f"Saved: {DATA_DIR / 'feature_cols.json'}")

print(f"Completed: {datetime.now()}")


Saved: data/features.parquet  shape=(806852, 814)
Saved: data/scaler.joblib
Saved: data/feature_cols.json
Completed: 2026-04-19 16:30:15.935770
